# Shared notebook runtime
Definitions for environment, logging, checkpoints, telemetry, plotting, and common training. This notebook has no training side effects.

In [ ]:
from pathlib import Path
import hashlib, json, logging, os, platform, random, subprocess, time
import numpy as np
import pandas as pd
import yaml
def project_root():
    root = Path.cwd().resolve()
    return root.parent if root.name == "notebooks" else root
def load_project_configs(root=None):
    root = Path(root or project_root())
    return {name: yaml.safe_load((root / "configs" / f"{name}.yaml").read_text(encoding="utf-8")) for name in ("data","models","training","backtest","regime")}
def seed_everything(seed):
    import torch
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

## Environment and provenance

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""): digest.update(chunk)
    return digest.hexdigest()
def environment_record(root=None, data_path=None):
    import importlib.metadata as metadata
    import torch
    root = Path(root or project_root())
    versions = {}
    for name in ("numpy","pandas","torch","PyYAML","mamba-ssm"):
        try: versions[name] = metadata.version(name)
        except metadata.PackageNotFoundError: versions[name] = None
    try: revision = subprocess.check_output(["git","rev-parse","HEAD"], cwd=root, text=True, stderr=subprocess.DEVNULL).strip()
    except (OSError, subprocess.CalledProcessError): revision = None
    return {"python":platform.python_version(),"platform":platform.platform(),"cuda":torch.cuda.is_available(),"packages":versions,"git_revision":revision,"data_sha256":sha256_file(data_path) if data_path else None}

## Logging, telemetry, and checkpoint helpers

In [ ]:
def configure_run_logging(run_dir):
    run_dir = Path(run_dir); run_dir.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger("financial_benchmark"); logger.setLevel(logging.INFO)
    target = (run_dir / "run.log").resolve()
    if not any(isinstance(h, logging.FileHandler) and Path(h.baseFilename) == target for h in logger.handlers):
        handler = logging.FileHandler(target, encoding="utf-8")
        handler.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
        logger.addHandler(handler)
    return logger
class RunTelemetry:
    def __init__(self): self.started = time.perf_counter(); self.events = []
    def mark(self, label, **values):
        event = {"label":label,"elapsed_seconds":time.perf_counter()-self.started,**values}
        self.events.append(event); return event
def save_training_checkpoint(path, model, optimizer, history, metadata=None):
    import torch
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"model_state":model.state_dict(),"optimizer_state":optimizer.state_dict(),"history":history,"metadata":metadata or {}}, path)
    return path
class NotebookUnitRun:
    def __init__(self, output_root, run_id, model, seed, fold, resume_mode="auto"):
        from src.runtime.experiment import atomic_json_dump
        self.atomic_json_dump = atomic_json_dump
        self.identity = {'model': model, 'seed': int(seed), 'fold': int(fold)}
        self.started = time.perf_counter()
        self.base = Path(output_root).resolve() / run_id
        self.unit = Path(model) / f"seed_{seed}" / f"fold_{fold}"
        self.checkpoints = self.base / "checkpoints" / model / f"seed_{seed}"
        self.telemetry = self.base / "telemetry" / self.unit
        self.table_dir = self.base / "tables" / self.unit
        self.checkpoints.mkdir(parents=True, exist_ok=True)
        self.telemetry.mkdir(parents=True, exist_ok=True)
        self.table_dir.mkdir(parents=True, exist_ok=True)
        self.state_path = self.telemetry / "state.json"
        self.events_path = self.telemetry / "events.jsonl"
        latest = self.checkpoints / "latest.pt"
        if latest.exists():
            import torch, shutil
            prior = torch.load(latest, map_location="cpu", weights_only=False)
            prior_identity = prior.get("unit_identity")
            if prior_identity != self.identity:
                if prior_identity is None: raise RuntimeError("Existing checkpoint lacks fold identity; preserve it manually before continuing")
                archive = self.checkpoints / f"fold_{prior_identity['fold']}"
                archive.mkdir(parents=True, exist_ok=True)
                for name in ("latest.pt", "best.pt"):
                    source = self.checkpoints / name
                    if source.exists(): shutil.copy2(source, archive / name)
                if resume_mode == "fail": raise FileExistsError("Checkpoint belongs to a different fold")
            elif resume_mode == "fail":
                raise FileExistsError("Checkpoint exists for this unit and RESUME_MODE=fail")
            elif resume_mode == "restart":
                import shutil
                from datetime import datetime, timezone
                stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
                archive = self.checkpoints / f"restart_{stamp}"
                archive.mkdir(parents=True, exist_ok=False)
                for name in ("latest.pt", "best.pt"):
                    source = self.checkpoints / name
                    if source.exists(): shutil.move(str(source), str(archive / name))
    def event(self, message, **fields):
        from datetime import datetime, timezone
        record = {"schema_version":"1.0","timestamp_utc":datetime.now(timezone.utc).isoformat(),"elapsed_seconds":time.perf_counter()-self.started,"run_id":self.base.name,**self.identity,"event":message,**fields}
        with self.events_path.open("a",encoding="utf-8") as stream: stream.write(json.dumps(record,default=str)+"\n")
        if message == 'epoch_complete':
            with (self.base/'telemetry'/'epoch_metrics.jsonl').open('a',encoding='utf-8') as stream: stream.write(json.dumps(record,default=str)+'\n')
    def save_state(self, state):
        self.atomic_json_dump(state,self.state_path)
    def save_checkpoint(self, payload, filename="latest.pt"):
        import torch, shutil
        target=self.checkpoints/filename
        temporary=target.with_suffix(target.suffix+".tmp")
        merged = {**dict(payload),"unit_identity":self.identity}
        merged["metadata"] = {**dict(getattr(self, 'checkpoint_metadata', {})), **dict(merged.get("metadata", {})), **self.identity}
        torch.save(merged,temporary)
        if target.exists(): shutil.copy2(target, self.checkpoints / f'previous_{filename}')
        os.replace(temporary,target)
        return target
    def load_checkpoint(self, map_location="cpu"):
        import torch
        saw_candidate = False
        for target in (self.checkpoints/"latest.pt", self.checkpoints/"previous_latest.pt"):
            if not target.exists(): continue
            saw_candidate = True
            try: payload = torch.load(target,map_location=map_location,weights_only=False)
            except Exception as exc:
                self.event('checkpoint_rejected',path=str(target),reason=repr(exc)); continue
            compatible = payload.get("unit_identity") == self.identity
            stored = payload.get('metadata', {}); current = getattr(self, 'checkpoint_metadata', {})
            for key in ('git_revision','data_manifest_hash','resolved_config_hash','dependency_lock_hash','image_identifier'):
                if current.get(key) is not None and stored.get(key) != current.get(key): compatible = False
            if compatible: return payload
            self.event('checkpoint_rejected',path=str(target),reason='incompatible unit identity')
        if saw_candidate: raise RuntimeError('No compatible valid checkpoint remains; refusing silent restart')
        return None


## Shared model and training orchestration

In [ ]:
def build_signal_model(name, lookback, num_features, num_assets, cfg):
    from src.models.dlinear import DLinear
    from src.models.lstm import LSTMSignalModel
    from src.models.patchtst import PatchTSTSignalModel
    if name == "dlinear": return DLinear(lookback,num_features,num_assets,trend_window=int(cfg["dlinear"]["trend_window"]))
    if name == "lstm":
        p=cfg["lstm"]; return LSTMSignalModel(lookback,num_features,num_assets,hidden_dim=int(p["hidden_dim"]),num_layers=int(p["layers"]),dropout=float(p["dropout"]))
    if name == "patchtst":
        p=cfg["patchtst"]; return PatchTSTSignalModel(lookback,num_features,num_assets,patch_length=int(p["patch_length"]),d_model=int(p["d_model"]),n_heads=int(p["heads"]),n_layers=int(p["layers"]))
    if name == "mamba":
        try: from src.models.mamba_model import MambaSignalModel
        except ImportError as exc: raise RuntimeError("Genuine Mamba requires mamba_ssm") from exc
        p=cfg["mamba"]; model=MambaSignalModel(lookback,num_features,num_assets,d_model=int(p["d_model"]),n_layers=int(p["layers"]))
        if getattr(model,"implementation_name",None)!="Mamba": raise RuntimeError("Mamba implementation identity check failed")
        return model
    raise ValueError(name)
def train_and_predict(name,seed,train_loader,val_loader,test_loader,shape,configs,run=None,resume=False,device='auto',epochs=None):
    import torch
    from src.losses.sharpe_loss import NegativeSharpeLoss
    from src.training.trainer import BaseTrainer
    seed_everything(seed)
    model=build_signal_model(name,*shape,configs["models"])
    tr=dict(configs["training"])
    tr["device"]=device
    if epochs is not None: tr["epochs"]=int(epochs)
    optimizer=torch.optim.Adam(model.parameters(),lr=float(tr["learning_rate"]))
    trainer=BaseTrainer(model,NegativeSharpeLoss(alpha=float(tr.get("loss_alpha",1.0))),optimizer,tr,run=run)
    history,seconds=trainer.train(train_loader,val_loader,resume=resume)
    if run is not None: run.save_checkpoint({"model_state":model.state_dict(),"optimizer_state":optimizer.state_dict(),"history":history},filename="best.pt")
    return model,optimizer,history,seconds,trainer.predict(test_loader)

## Plotting helpers

In [ ]:
def plot_training_history(history,title="Training loss"):
    import matplotlib.pyplot as plt
    fig,ax=plt.subplots(figsize=(7,4))
    for key in ("train_loss","val_loss"):
        if history.get(key): ax.plot(history[key],label=key)
    ax.set(title=title,xlabel="Epoch",ylabel="Loss"); ax.legend()
    return fig,ax
def plot_rolling_sharpe(returns,window=63):
    import matplotlib.pyplot as plt
    series=pd.Series(returns).dropna()
    value=series.rolling(window).mean()/series.rolling(window).std()*np.sqrt(252)
    fig,ax=plt.subplots(figsize=(9,4)); value.plot(ax=ax)
    ax.set(title=f"Rolling {window}-period Sharpe",ylabel="Sharpe")
    return fig,ax